In [1]:
# ============================================================
# 03_supervised_source_train.ipynb
# Source supervised training on A only
# Hybrid model:
#   - phase branch = main spatial cue
#   - amplitude branch = auxiliary correction
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = Path("/home/tonyliao/Location")
BUILD_DIR = DATA_ROOT / "dataset_build_hybrid"
SSL_DIR   = DATA_ROOT / "ssl_pretrain_runs"
OUT_DIR   = DATA_ROOT / "source_train_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 64
EPOCHS = 150
LR = 1e-4

TARGET_T = 256
TARGET_S = None   # infer from first sample

USE_SSL_ENCODER = True
SSL_ENCODER_PATH = SSL_DIR / "amp_ssl_encoder_best.keras"

PHASE_BRANCH_TRAINABLE = True
FREEZE_SSL_BACKBONE_EPOCH0 = True
UNFREEZE_SSL_LAST_N = 4

# loss weights
W_PRES   = 1.0
W_CLASS  = 1.0
W_COARSE = 1.0
W_DELTA  = 0.25
W_FINAL  = 1.5

# Full coordinate map for all possible known labels
# Keep this complete so label_map validation is safe.
CLASS_CENTER_MAP = {
    "Empty": [0.0, 0.0],
    # "Center": [3.0, 4.0],
    # "Corner": [0.0, 8.0],
    "Corner_LeftDown": [1.0, 7.0],
    "Corner_LeftUp": [1.0, 1.0],
    "Corner_RightDown": [5.0, 7.0],
    "Corner_RightUp": [5.0, 1.0],
    # "Empty_Pred": [0.0, 0.0],
    "LeftDown": [2.0, 6.0],
    # "LeftDown_Far": [1.5, 7.0],
    "LeftMid": [2.0, 4.0],
    "LeftUp": [2.0, 2.0],
    # "LeftUp_Near": [2.0, 2.5],
    # "LeftUp_Pred": [2.0, 2.0],
    "MiddleDown": [3.0, 6.0],
    "MiddleUp": [3.0, 2.0],
    "RightDown": [4.0, 6.0],
    "RightMid": [4.0, 4.0],
    "RightUp": [4.0, 2.0],
    # "Wall": [6.0, 5.0],
}

# -----------------------------
# Load dataset
# -----------------------------
def load_npz(path: Path):
    obj = np.load(path, allow_pickle=True)
    return {k: obj[k] for k in obj.files}

A_train = load_npz(BUILD_DIR / "A_train.npz")
A_val   = load_npz(BUILD_DIR / "A_val.npz")

with open(BUILD_DIR / "label_map.json", "r", encoding="utf-8") as f:
    label_meta = json.load(f)

label_map = label_meta["label_map"]
inv_label_map = {int(k): v for k, v in label_meta["inv_label_map"].items()}
num_classes = len(label_map)

print("A_train:", len(A_train["amp_path"]))
print("A_val:", len(A_val["amp_path"]))
print("num_classes:", num_classes)

if len(A_train["amp_path"]) == 0:
    raise ValueError("A_train is empty.")
if len(A_val["amp_path"]) == 0:
    raise ValueError("A_val is empty.")

missing_centers = sorted([
    lab for lab in label_map.keys()
    if lab not in CLASS_CENTER_MAP
])
if missing_centers:
    raise ValueError(
        f"These labels exist in label_map but not in CLASS_CENTER_MAP: {missing_centers}"
    )

# -----------------------------
# Helpers
# -----------------------------
def ensure_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x[..., None]
    return x

def resize_to_target(x, target_t, target_s):
    x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
    x_tf = tf.image.resize(x_tf, size=(target_t, target_s), method="bilinear")
    return x_tf.numpy().astype(np.float32)

def zscore_per_sample(x):
    mu = np.mean(x, axis=(0, 1), keepdims=True)
    sd = np.std(x, axis=(0, 1), keepdims=True) + 1e-6
    return ((x - mu) / sd).astype(np.float32)

def load_amp(path):
    x = np.load(str(path)).astype(np.float32)
    x = ensure_3d(x)
    return x

def load_pha(path, ref_shape=None):
    if path is None or str(path) == "":
        if ref_shape is None:
            raise ValueError("pha_path missing and ref_shape is None")
        return np.zeros(ref_shape, dtype=np.float32)

    p = Path(str(path))
    if not p.exists():
        if ref_shape is None:
            raise ValueError(f"pha_path does not exist: {path}")
        return np.zeros(ref_shape, dtype=np.float32)

    x = np.load(str(p)).astype(np.float32)
    x = ensure_3d(x)
    return x

sample_amp = load_amp(A_train["amp_path"][0])
if TARGET_S is None:
    TARGET_S = sample_amp.shape[1]

print("TARGET_T =", TARGET_T)
print("TARGET_S =", TARGET_S)

def preprocess_amp(path):
    x = load_amp(path)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def preprocess_pha(path, amp_shape=None):
    x = load_pha(path, ref_shape=amp_shape)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def onehot(num_classes, y):
    v = np.zeros((num_classes,), dtype=np.float32)
    v[int(y)] = 1.0
    return v

def xy_from_label(label_name):
    label_name = str(label_name)
    if label_name not in CLASS_CENTER_MAP:
        raise KeyError(f"Label {label_name} not found in CLASS_CENTER_MAP")
    return np.asarray(CLASS_CENTER_MAP[label_name], dtype=np.float32)

# -----------------------------
# Dataset sequence
# -----------------------------
class HybridSequence(keras.utils.Sequence):
    def __init__(self, obj, batch_size=16, shuffle=True):
        self.obj = obj
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(obj["amp_path"]))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        inds = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]

        x_amp = []
        x_pha = []

        y_presence = []
        y_class = []
        y_coarse = []
        y_delta = []
        y_final = []

        sw_presence = []
        sw_class = []
        sw_coarse = []
        sw_delta = []
        sw_final = []

        for i in inds:
            amp = preprocess_amp(self.obj["amp_path"][i])
            pha = preprocess_pha(self.obj["pha_path"][i], amp_shape=amp.shape)

            label_name = str(self.obj["anchor_label"][i])
            label_id = int(self.obj["label_id"][i])
            presence = int(self.obj["presence"][i])

            if presence == 1:
                xy = xy_from_label(label_name)
            else:
                xy = np.zeros((2,), dtype=np.float32)

            # source-stage target: coarse target equals known anchor center
            coarse_xy = xy.copy()

            # residual target is zero at source stage; later stages can learn adaptation
            delta_xy = np.zeros((2,), dtype=np.float32)

            x_amp.append(amp)
            x_pha.append(pha)

            y_presence.append([presence])
            y_class.append(onehot(num_classes, label_id))
            y_coarse.append(coarse_xy)
            y_delta.append(delta_xy)
            y_final.append(xy)

            sw_presence.append(1.0)
            sw_class.append(1.0)

            loc_w = 1.0 if presence == 1 else 0.0
            sw_coarse.append(loc_w)
            sw_delta.append(loc_w)
            sw_final.append(loc_w)

        x = {
            "amp_in": np.stack(x_amp).astype(np.float32),
            "pha_in": np.stack(x_pha).astype(np.float32),
        }

        y = {
            "presence_out": np.stack(y_presence).astype(np.float32),
            "class_out": np.stack(y_class).astype(np.float32),
            "coarse_xy_out": np.stack(y_coarse).astype(np.float32),
            "amp_delta_out": np.stack(y_delta).astype(np.float32),
            "final_xy_out": np.stack(y_final).astype(np.float32),
        }

        sw = {
            "presence_out": np.asarray(sw_presence, dtype=np.float32),
            "class_out": np.asarray(sw_class, dtype=np.float32),
            "coarse_xy_out": np.asarray(sw_coarse, dtype=np.float32),
            "amp_delta_out": np.asarray(sw_delta, dtype=np.float32),
            "final_xy_out": np.asarray(sw_final, dtype=np.float32),
        }

        return x, y, sw

train_seq = HybridSequence(A_train, batch_size=BATCH_SIZE, shuffle=True)
val_seq   = HybridSequence(A_val, batch_size=BATCH_SIZE, shuffle=False)

bx, by, bsw = train_seq[0]
print("amp_in:", bx["amp_in"].shape)
print("pha_in:", bx["pha_in"].shape)
for k, v in by.items():
    print(k, v.shape)

# -----------------------------
# Model
# -----------------------------
def conv_block(x, filters, pool=True, dropout=0.0):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    if pool:
        x = layers.MaxPooling2D(pool_size=2)(x)
    if dropout > 0:
        x = layers.Dropout(dropout)(x)
    return x

def build_phase_encoder(input_shape):
    inp = keras.Input(shape=input_shape, name="pha_geom_input")
    x = conv_block(inp, 32, pool=True, dropout=0.05)
    x = conv_block(x, 64, pool=True, dropout=0.05)
    x = conv_block(x, 128, pool=True, dropout=0.10)
    x = conv_block(x, 256, pool=True, dropout=0.10)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    feat = layers.Dense(128, activation="relu", name="pha_feat")(x)
    return keras.Model(inp, feat, name="phase_geometry_encoder")

def build_amp_encoder(input_shape):
    inp = keras.Input(shape=input_shape, name="amp_in_local")
    x = conv_block(inp, 32, pool=True, dropout=0.05)
    x = conv_block(x, 64, pool=True, dropout=0.05)
    x = conv_block(x, 128, pool=True, dropout=0.10)
    x = conv_block(x, 256, pool=True, dropout=0.10)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    feat = layers.Dense(128, activation="relu", name="amp_feat")(x)
    return keras.Model(inp, feat, name="amp_encoder_local")

def build_hybrid_model(amp_input_shape, pha_input_shape, num_classes):
    amp_in = keras.Input(shape=amp_input_shape, name="amp_in")
    pha_in = keras.Input(shape=pha_input_shape, name="pha_in")

    # amplitude branch
    if USE_SSL_ENCODER and SSL_ENCODER_PATH.exists():
        ssl_encoder = keras.models.load_model(SSL_ENCODER_PATH, compile=False)
        amp_embed = ssl_encoder(amp_in)
        amp_feat = layers.Dense(128, activation="relu", name="amp_feat")(amp_embed)

        if FREEZE_SSL_BACKBONE_EPOCH0:
            for layer in ssl_encoder.layers:
                layer.trainable = False
            if UNFREEZE_SSL_LAST_N > 0:
                for layer in ssl_encoder.layers[-UNFREEZE_SSL_LAST_N:]:
                    layer.trainable = True
    else:
        ssl_encoder = None
        amp_feat = build_amp_encoder(amp_input_shape)(amp_in)

    # phase branch
    pha_encoder = build_phase_encoder(pha_input_shape)
    pha_encoder.trainable = PHASE_BRANCH_TRAINABLE
    pha_feat = pha_encoder(pha_in)

    # presence head from amplitude
    presence_feat = layers.Dense(64, activation="relu")(amp_feat)
    presence_out = layers.Dense(1, activation="sigmoid", name="presence_out")(presence_feat)

    # fusion
    fused = layers.Concatenate(name="single_receiver_feature_fusion")([pha_feat, amp_feat])
    fused = layers.Dense(128, activation="relu")(fused)
    fused = layers.Dropout(0.10)(fused)

    # class head
    class_out = layers.Dense(num_classes, activation="softmax", name="class_out")(fused)

    # fixed class-center lookup via frozen dense
    center_matrix = np.zeros((num_classes, 2), dtype=np.float32)
    for class_id in range(num_classes):
        label_name = inv_label_map[class_id]
        center_matrix[class_id] = xy_from_label(label_name)

    coarse_xy_layer = layers.Dense(
        2,
        use_bias=False,
        trainable=False,
        name="coarse_xy_out"
    )
    coarse_xy_out = coarse_xy_layer(class_out)

    # auxiliary amplitude correction
    corr = layers.Dense(64, activation="relu")(amp_feat)
    raw_delta = layers.Dense(2, activation="tanh")(corr)
    amp_delta_out = layers.Rescaling(scale=0.75, name="amp_delta_out")(raw_delta)

    # learned gate for correction
    gate = layers.Concatenate(name="delta_gate_fusion")([amp_feat, pha_feat, presence_out])
    gate = layers.Dense(32, activation="relu")(gate)
    gate = layers.Dense(1, activation="sigmoid", name="delta_gate")(gate)

    scaled_delta = layers.Multiply(name="scaled_delta")([amp_delta_out, gate])
    final_xy_out = layers.Add(name="final_xy_out")([coarse_xy_out, scaled_delta])

    model = keras.Model(
        inputs={"amp_in": amp_in, "pha_in": pha_in},
        outputs={
            "presence_out": presence_out,
            "class_out": class_out,
            "coarse_xy_out": coarse_xy_out,
            "amp_delta_out": amp_delta_out,
            "final_xy_out": final_xy_out,
        },
        name="hybrid_source_A_model"
    )

    model.get_layer("coarse_xy_out").set_weights([center_matrix])
    return model

amp_input_shape = bx["amp_in"].shape[1:]
pha_input_shape = bx["pha_in"].shape[1:]
model = build_hybrid_model(amp_input_shape, pha_input_shape, num_classes)
model.summary()

# -----------------------------
# Compile / train
# -----------------------------
losses = {
    "presence_out": keras.losses.BinaryCrossentropy(),
    "class_out": keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    "coarse_xy_out": keras.losses.Huber(delta=0.5),
    "amp_delta_out": keras.losses.Huber(delta=0.25),
    "final_xy_out": keras.losses.Huber(delta=0.5),
}

loss_weights = {
    "presence_out": W_PRES,
    "class_out": W_CLASS,
    "coarse_xy_out": W_COARSE,
    "amp_delta_out": W_DELTA,
    "final_xy_out": W_FINAL,
}

metrics = {
    "presence_out": [
        keras.metrics.BinaryAccuracy(name="acc"),
        keras.metrics.Precision(name="prec"),
        keras.metrics.Recall(name="rec"),
    ],
    "class_out": [
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.TopKCategoricalAccuracy(k=2, name="top2"),
    ],
    "coarse_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "amp_delta_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "final_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
}

model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_source_A_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "source_A_train_log.csv")),
]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

model.save(OUT_DIR / "hybrid_source_A_final.keras")

with open(OUT_DIR / "source_A_train_history.json", "w", encoding="utf-8") as f:
    json.dump(history.history, f, indent=2)

summary = {
    "training_domain": "A",
    "single_receiver_input_only": True,
    "fused_AB_input": False,
    "phase_branch_main_spatial_cue": True,
    "amplitude_branch_auxiliary": True,
    "use_ssl_encoder": bool(USE_SSL_ENCODER and SSL_ENCODER_PATH.exists()),
    "ssl_encoder_path": str(SSL_ENCODER_PATH),
    "freeze_ssl_backbone_epoch0": bool(FREEZE_SSL_BACKBONE_EPOCH0),
    "unfreeze_ssl_last_n": int(UNFREEZE_SSL_LAST_N),
    "phase_branch_trainable": bool(PHASE_BRANCH_TRAINABLE),
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LR,
    "target_t": TARGET_T,
    "target_s": int(TARGET_S),
    "A_train_windows": int(len(A_train["amp_path"])),
    "A_val_windows": int(len(A_val["amp_path"])),
    "num_classes": int(num_classes),
    "source_labels": sorted(list(set(map(str, A_train["anchor_label"].tolist())))),
    "monitor": "val_loss",
}

with open(OUT_DIR / "source_A_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved to:", OUT_DIR)
print(json.dumps(summary, indent=2))

2026-04-13 14:18:57.701021: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-13 14:18:57.707406: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776089937.714831 1571628 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776089937.717063 1571628 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776089937.722815 1571628 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

A_train: 2969
A_val: 742
num_classes: 13
TARGET_T = 256
TARGET_S = 41


I0000 00:00:1776089938.915859 1571628 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


amp_in: (64, 256, 41, 2)
pha_in: (64, 256, 41, 2)
presence_out (64, 1)
class_out (64, 13)
coarse_xy_out (64, 2)
amp_delta_out (64, 2)
final_xy_out (64, 2)


Model: "hybrid_source_A_model"

...

 4/47 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - amp_delta_out_loss: 0.0323 - amp_delta_out_mae: 0.2437 - class_out_acc: 0.1227 - class_out_loss: 2.6478 - class_out_top2: 0.2464 - coarse_xy_out_loss: 0.6004 - coarse_xy_out_mae: 1.7376 - final_xy_out_loss: 0.6011 - final_xy_out_mae: 1.7479 - loss: 5.2589 - presence_out_acc: 0.1663 - presence_out_loss: 1.1011 - presence_out_prec: 0.9411 - presence_out_rec: 0.0743

2026-04-13 14:19:14.463892: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_9429', 20 bytes spill stores, 20 bytes spill loads

2026-04-13 14:19:15.304963: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_9433', 32 bytes spill stores, 32 bytes spill loads

2026-04-13 14:19:15.314131: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_9431', 32 bytes spill stores, 32 bytes spill loads

2026-04-13 14:19:15.332087: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_9429', 32 bytes spill stores, 32 bytes spill loads

2026-04-13 14:19:15.707323: I external/l

47/47 ━━━━━━━━━━━━━━━━━━━━ 0s 290ms/step - amp_delta_out_loss: 0.0304 - amp_delta_out_mae: 0.2291 - class_out_acc: 0.4589 - class_out_loss: 1.9321 - class_out_top2: 0.6037 - coarse_xy_out_loss: 0.5119 - coarse_xy_out_mae: 1.4115 - final_xy_out_loss: 0.5085 - final_xy_out_mae: 1.4098 - loss: 3.9538 - presence_out_acc: 0.4990 - presence_out_loss: 0.7685 - presence_out_prec: 0.9318 - presence_out_rec: 0.4870

2026-04-13 14:19:29.648529: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_437', 28 bytes spill stores, 28 bytes spill loads

2026-04-13 14:19:29.731511: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_451', 128 bytes spill stores, 128 bytes spill loads

2026-04-13 14:19:30.491131: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_444', 152 bytes spill stores, 152 bytes spill loads




Epoch 1: val_loss improved from inf to 4.19135, saving model to /home/tonyliao/Location/source_train_runs/hybrid_source_A_best.keras
47/47 ━━━━━━━━━━━━━━━━━━━━ 32s 424ms/step - amp_delta_out_loss: 0.0304 - amp_delta_out_mae: 0.2290 - class_out_acc: 0.4638 - class_out_loss: 1.9201 - class_out_top2: 0.6080 - coarse_xy_out_loss: 0.5094 - coarse_xy_out_mae: 1.4046 - final_xy_out_loss: 0.5060 - final_xy_out_mae: 1.4028 - loss: 3.9304 - presence_out_acc: 0.5040 - presence_out_loss: 0.7636 - presence_out_prec: 0.9317 - presence_out_rec: 0.4931 - val_amp_delta_out_loss: 0.0297 - val_amp_delta_out_mae: 0.2277 - val_class_out_acc: 0.1631 - val_class_out_loss: 2.4202 - val_class_out_top2: 0.2857 - val_coarse_xy_out_loss: 0.6061 - val_coarse_xy_out_mae: 1.6903 - val_final_xy_out_loss: 0.5939 - val_final_xy_out_mae: 1.6677 - val_loss: 4.1914 - val_presence_out_acc: 0.9205 - val_presence_out_loss: 0.2673 - val_presence_out_prec: 0.9205 - val_presence_out_rec: 1.0000 - learning_rate: 1.0000e-04
Epoc